In [1]:
import pandas as pd
import numpy as np

CURRENT_YEAR = 2026

# ============================================================
# LOAD FILES
# ============================================================

df = pd.read_csv("data/lexus.csv")
dep_df = pd.read_csv("data/annual_dep_rate.csv")

# ============================================================
# CLEAN DEPRECIATION DATA
# ============================================================

dep_df["make"] = dep_df["make"].astype(str).str.strip().str.upper()
dep_df["model"] = dep_df["model"].astype(str).str.strip().str.upper()

# ============================================================
# LEXUS MODEL MAPPING
# ============================================================

MODEL_MAPPING = {
    "lexus-es": "ES",
    "lexus-gs": "GS",
    "lexus-gx": "GX",
    "lexus-is": "IS",
    "lexus-lc": "LC",
    "lexus-ls": "LS",
    "lexus-lx": "LX",
    "lexus-nx": "NX",
    "lexus-rx": "RX",
    "lexus-rz": "RZ",
    "lexus-ux": "UX",
}

df["make"] = "LEXUS"

df["model_name"] = (
    df["model_slug"]
    .map(MODEL_MAPPING)
    .astype(str)
    .str.upper()
)

# ============================================================
# BUILD LOOKUP
# ============================================================

dep_lookup = (
    dep_df.groupby(["make", "model"])["annual_dep_rate"]
    .mean()
    .reset_index()
)

# ============================================================
# MERGE DEPRECIATION RATE
# ============================================================

df = df.merge(
    dep_lookup,
    left_on=["make", "model_name"],
    right_on=["make", "model"],
    how="left"
)

# ============================================================
# CAR AGE
# ============================================================

df["car_age"] = CURRENT_YEAR - df["year"]
df["car_age"] = df["car_age"].clip(lower=0)

# ============================================================
# DEPRECIATED VALUE
# ============================================================

def calc_depreciated_value(row):

    rate = row["annual_dep_rate"]

    if pd.isna(rate):
        return np.nan

    age = row["car_age"]

    if age == 0:
        return round(row["price_avg_aed"], 0)

    return round(
        row["price_avg_aed"] * ((1 - rate) ** age),
        0
    )

df["depreciated_value"] = df.apply(
    calc_depreciated_value,
    axis=1
)

# ============================================================
# VALIDATION
# ============================================================

print("Total Rows:", len(df))
print("Matched Rates:", df["annual_dep_rate"].notna().sum())
print("Missing Rates:", df["annual_dep_rate"].isna().sum())

print("\nUnmatched Models:")
print(
    df[df["annual_dep_rate"].isna()]
    ["model_slug"]
    .drop_duplicates()
    .tolist()
)

# ============================================================
# SAVE
# ============================================================

df.to_csv(
    "data/lexus_dep.csv",
    index=False
)

print("\nSaved: data/lexus_dep.csv")

Total Rows: 136
Matched Rates: 22
Missing Rates: 114

Unmatched Models:
['lexus-gx', 'lexus-is', 'lexus-lc', 'lexus-ls', 'lexus-lx', 'lexus-nx', 'lexus-rc', 'lexus-rc-f', 'lexus-rx', 'lexus-rx-hybrid', 'lexus-ux', 'lexus-ux-hybrid']

Saved: data/lexus_dep.csv
